# Data Transformation Assignment

**Objective:** Prepare the Employee Productivity dataset for Machine Learning by applying:
- Encoding
- Normalization
- Standardization (Scaling)
- Preprocessing Pipeline

**Dataset:** `Employee_productivity_Dataset`

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Upload the Employee Productivity dataset in Google Colab
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))

if file_name.lower().endswith('.csv'):
    df = pd.read_csv(file_name)
elif file_name.lower().endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_name)
else:
    raise ValueError("Please upload the Employee Productivity dataset as CSV or Excel.")

print("Dataset Shape:", df.shape)
display(df.head())
print("\nColumn Information:")
df.info()

## Q1. Handle Missing Values (Basic Cleaning)

**Question:** Fill missing values using the following:
- Age → Median
- Salary → Mean
- Hours_Worked_Per_Week → Median
- Performance_Score → Mean

Display the dataset after handling missing values.

### Solution
Median is used for Age and Hours Worked because it is less affected by extreme values. Mean is used for Salary and Performance Score as required in the question.

In [ ]:
# Missing values before cleaning
print("Missing values before handling:")
display(df[['Age', 'Salary', 'Hours_Worked_Per_Week', 'Performance_Score']].isnull().sum())

# Fill missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Salary'] = df['Salary'].fillna(df['Salary'].mean())
df['Hours_Worked_Per_Week'] = df['Hours_Worked_Per_Week'].fillna(
    df['Hours_Worked_Per_Week'].median()
)
df['Performance_Score'] = df['Performance_Score'].fillna(
    df['Performance_Score'].mean()
)

print("Missing values after handling:")
display(df[['Age', 'Salary', 'Hours_Worked_Per_Week', 'Performance_Score']].isnull().sum())

print("Dataset after handling missing values:")
display(df)

## Q2. Label Encoding

**Question:** Convert these columns into numeric values using Label Encoding:
- Gender
- Department

Show the updated columns.

### Solution
Label Encoding assigns a unique integer to each category.

In [ ]:
df_label = df.copy()

label_encoders = {}

for column in ['Gender', 'Department']:
    le = LabelEncoder()
    df_label[column] = le.fit_transform(df_label[column].astype(str))
    label_encoders[column] = le

    print(f"{column} Mapping:")
    print(dict(zip(le.classes_, le.transform(le.classes_))))

print("\nUpdated encoded columns:")
display(df_label[['Gender', 'Department']].head(10))

## Q3. One-Hot Encoding

**Question:** Apply One-Hot Encoding on:
- Work_Mode
- Location

Display the dataset and check how many new columns are created.

### Solution
One-Hot Encoding creates separate binary columns for categories and avoids implying an artificial numerical order.

In [ ]:
columns_to_encode = ['Work_Mode', 'Location']

original_column_count = df.shape[1]

df_onehot = pd.get_dummies(
    df,
    columns=columns_to_encode,
    prefix=columns_to_encode,
    dtype=int
)

new_column_count = df_onehot.shape[1]
new_columns_created = new_column_count - original_column_count + len(columns_to_encode)

print("Original number of columns:", original_column_count)
print("Number of one-hot encoded columns created:", new_columns_created)
print("Final number of columns:", new_column_count)

display(df_onehot.head())

## Q4. Normalization (Min-Max Scaling)

**Question:** Normalize the following columns:
- Salary
- Hours_Worked_Per_Week

Use `MinMaxScaler` and display the results.

### Solution
Min-Max Scaling transforms values to the range 0 to 1.

In [ ]:
df_minmax = df.copy()

minmax_scaler = MinMaxScaler()

df_minmax[['Salary', 'Hours_Worked_Per_Week']] = minmax_scaler.fit_transform(
    df_minmax[['Salary', 'Hours_Worked_Per_Week']]
)

print("Normalized values:")
display(df_minmax[['Salary', 'Hours_Worked_Per_Week']].head(10))

print("Minimum values:")
display(df_minmax[['Salary', 'Hours_Worked_Per_Week']].min())

print("Maximum values:")
display(df_minmax[['Salary', 'Hours_Worked_Per_Week']].max())

## Q5. Standardization (Scaling)

**Question:** Apply `StandardScaler` on:
- Age
- Projects_Completed

Display the transformed values.

### Solution
Standardization converts the data so that each transformed feature has approximately mean 0 and standard deviation 1.

In [ ]:
df_standard = df.copy()

standard_scaler = StandardScaler()

df_standard[['Age', 'Projects_Completed']] = standard_scaler.fit_transform(
    df_standard[['Age', 'Projects_Completed']]
)

print("Standardized values:")
display(df_standard[['Age', 'Projects_Completed']].head(10))

print("Means after standardization:")
display(df_standard[['Age', 'Projects_Completed']].mean())

print("Standard deviations after standardization:")
display(df_standard[['Age', 'Projects_Completed']].std(ddof=0))

## Q6. Compare Scaling Methods

**Question:** Apply both:
- MinMaxScaler
- StandardScaler

on the Salary column. Show both results side by side.

### Solution
MinMaxScaler places Salary between 0 and 1, while StandardScaler expresses each Salary in terms of its distance from the mean measured in standard deviations.

In [ ]:
salary_comparison = pd.DataFrame({
    'Original_Salary': df['Salary']
})

salary_comparison['MinMax_Salary'] = MinMaxScaler().fit_transform(
    df[['Salary']]
).ravel()

salary_comparison['Standardized_Salary'] = StandardScaler().fit_transform(
    df[['Salary']]
).ravel()

print("Comparison of scaling methods:")
display(salary_comparison.head(15))

## Q7. Build Preprocessing Pipeline

**Question:** Create a pipeline that:
- Applies encoding to categorical columns
- Applies scaling to numerical columns

Use:
- `ColumnTransformer`
- `Pipeline`

### Solution
A preprocessing pipeline makes the transformation process consistent and reusable. Categorical variables are imputed and one-hot encoded, while numerical variables are imputed and standardized.

In [ ]:
# Automatically identify numerical and categorical columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical Columns:", numeric_columns)
print("Categorical Columns:", categorical_columns)

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_columns),
        ('cat', categorical_pipeline, categorical_columns)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

print("\nPreprocessing Pipeline:")
print(pipeline)

## Q8. Apply Pipeline

**Question:** Apply the pipeline on the dataset. Display:
- Transformed dataset
- Shape of final dataset

### Solution
The complete preprocessing pipeline is now applied to the Employee Productivity dataset.

In [ ]:
transformed_data = pipeline.fit_transform(df)

# Convert sparse output to dense when necessary
if hasattr(transformed_data, 'toarray'):
    transformed_array = transformed_data.toarray()
else:
    transformed_array = transformed_data

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

transformed_df = pd.DataFrame(
    transformed_array,
    columns=feature_names
)

print("Transformed Dataset:")
display(transformed_df.head(10))

print("Original dataset shape:", df.shape)
print("Final transformed dataset shape:", transformed_df.shape)

## Q9. Conceptual Question

**Question:** Why is scaling important in Python?

### Solution

Scaling is important because different numerical features can have very different ranges. For example, Salary may contain values in thousands while Age may contain values in tens. Without scaling, features with larger numerical values can have a greater influence on many Machine Learning algorithms.

Scaling helps features contribute more fairly, improves numerical stability, and can help algorithms such as KNN, K-Means, SVM and gradient-based models train more effectively.

## Q10. Conceptual Question

**Question:** Why do we convert categorical data into numerical form?

### Solution

Most Machine Learning algorithms work with numerical values and cannot directly process text categories such as Male/Female, Department names, Work Mode or Location.

Therefore, categorical data is converted into numerical form using methods such as Label Encoding and One-Hot Encoding. This allows Machine Learning models to use categorical information during training while keeping the data in a mathematical form that the algorithms can process.

# Conclusion

The Employee Productivity dataset has been prepared for Machine Learning using missing-value treatment, Label Encoding, One-Hot Encoding, Min-Max Normalization, Standardization, scaling comparison, and a complete preprocessing pipeline using `ColumnTransformer` and `Pipeline`.